# Лаба 2: KNN
Классификация на Fashion-MNIST (KNN) и регрессия на Insurance (KNN).


In [1]:

import sys, os
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.environ.setdefault('MPLCONFIGDIR', 'outputs/mpl_cache')
os.makedirs('outputs/mpl_cache', exist_ok=True)

from src.models.knn import KNN

torch.manual_seed(42)
np.random.seed(42)


## Классификация: Fashion-MNIST (подвыборка)


In [2]:

# Классификация Fashion-MNIST (k=5) с train/val сплитом
import pandas as pd

raw = pd.read_csv('../datasets/fashionmnist/fashion-mnist_train.csv')
raw = raw.sample(n=min(len(raw), 8000), random_state=42).reset_index(drop=True)
labels = raw['label'].astype('int64')
X = raw.drop(columns=['label']).astype('float32') / 255.0
X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(labels.values, dtype=torch.long)

# стандартизация
mean = X_tensor.mean(dim=0, keepdim=True)
std = X_tensor.std(dim=0, keepdim=True)
std[std < 1e-6] = 1.0
X_std = (X_tensor - mean) / std

# train/val/test: 80/10/10

def train_val_test_split(X, y, val_ratio=0.1, test_ratio=0.1, seed=42):
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0]
    idx = torch.randperm(n, generator=g)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    test_idx = idx[:n_test]; val_idx = idx[n_test:n_test+n_val]; train_idx = idx[n_test+n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx], X[test_idx], y[test_idx]

Xtr, ytr, Xval, yval, Xte, yte = train_val_test_split(X_std, y_tensor, seed=42)
print(f'Train: {Xtr.shape}, Val: {Xval.shape}, Test: {Xte.shape}')

knn_clf = KNN(task='classification', k=5, p=2, weights='uniform')
knn_clf.fit(Xtr, ytr)

with torch.no_grad():
    y_pred_train = knn_clf.predict(Xtr)
    y_pred_val = knn_clf.predict(Xval)

train_acc = (y_pred_train == ytr.cpu()).float().mean().item()
val_acc = (y_pred_val == yval.cpu()).float().mean().item()
print(f'Train acc={train_acc:.4f} | Val acc={val_acc:.4f}')


Train: torch.Size([6400, 784]), Val: torch.Size([800, 784]), Test: torch.Size([800, 784])
Train acc=0.8686 | Val acc=0.8037


## Регрессия: Insurance (predict charges)


In [3]:

# Регрессия на insurance (80/10/10, k=10) с RMSE
import pandas as pd
import torch

# читаем полный датасет и стандартизируем признаки
raw = pd.read_csv('../datasets/insurance.csv')
X = raw.drop(columns=['charges'])
y = raw['charges']
X_enc = pd.get_dummies(X, drop_first=True).astype('float32')
X_tensor = torch.tensor(X_enc.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32)
mean = X_tensor.mean(dim=0, keepdim=True)
std = X_tensor.std(dim=0, keepdim=True) + 1e-6
X_std = (X_tensor - mean) / std

# 80/10/10 split

def train_val_test_split(X, y, val_ratio=0.1, test_ratio=0.1, seed=42):
    g = torch.Generator().manual_seed(seed)
    n = X.shape[0]
    idx = torch.randperm(n, generator=g)
    n_test = int(n * test_ratio)
    n_val = int(n * val_ratio)
    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx], X[test_idx], y[test_idx]

Xtr, ytr, Xval, yval, Xte, yte = train_val_test_split(X_std, y_tensor, seed=42)
print(f'Train: {Xtr.shape}, Val: {Xval.shape}, Test: {Xte.shape}')

knn_reg = KNN(task='regression', k=10, p=2, weights='uniform')
knn_reg.fit(Xtr, ytr)

with torch.no_grad():
    preds_train = knn_reg.predict(Xtr)
    preds_val = knn_reg.predict(Xval)
    preds_test = knn_reg.predict(Xte)

rmse = lambda a, b: torch.sqrt(((a - b) ** 2).mean()).item()
print('Results (k=10):')
print('  Train RMSE:', rmse(preds_train, ytr.cpu()))
print('  Val   RMSE:', rmse(preds_val, yval.cpu()))
print('  Test  RMSE:', rmse(preds_test, yte.cpu()))


Train: torch.Size([1072, 8]), Val: torch.Size([133, 8]), Test: torch.Size([133, 8])
Results (k=10):
  Train RMSE: 5068.12353515625
  Val   RMSE: 5625.1044921875
  Test  RMSE: 4517.98583984375
